# TP1 - Panoramica

In [7]:
import numpy as np
import cv2

### Conjunto de imágenes

In [ ]:
# Cuadros
L_cuadro = cv2.imread('img/inputs/cuadro_0.jpg', cv2.IMREAD_COLOR)
C_cuadro = cv2.imread('img/inputs/cuadro_1.jpg', cv2.IMREAD_COLOR)
R_cuadro = cv2.imread('img/inputs/cuadro_2.jpg', cv2.IMREAD_COLOR)

# Udesa
L_udesa = cv2.imread('img/inputs/udesa_0.jpg', cv2.IMREAD_COLOR)
C_udesa = cv2.imread('img/inputs/udesa_1.jpg', cv2.IMREAD_COLOR)
R_udesa = cv2.imread('img/inputs/udesa_2.jpg', cv2.IMREAD_COLOR)

def resize(img, w):
    w_, h_ = img.shape[1], img.shape[0]
    s = w / w_                  # factor de escala
    h = int(h_ * s)
    resized = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    return resized, s

L_cuadro_resize, sL = resize(L_cuadro, 600)
C_cuadro_resize, sC = resize(C_cuadro, 600)
R_cuadro_resize, sR = resize(R_cuadro, 600)

L_udesa_resize, sL = resize(L_udesa, 600)
C_udesa_resize, sC = resize(C_udesa, 600)
R_udesa_resize, sR = resize(R_udesa, 600)

# guardalas en img/inputs-resized
cv2.imwrite('img/inputs-resized/L_cuadro.jpg', L_cuadro_resize)
cv2.imwrite('img/inputs-resized/C_cuadro.jpg', C_cuadro_resize)
cv2.imwrite('img/inputs-resized/R_cuadro.jpg', R_cuadro_resize)
cv2.imwrite('img/inputs-resized/L_udesa.jpg', L_udesa_resize)
cv2.imwrite('img/inputs-resized/C_udesa.jpg', C_udesa_resize)
cv2.imwrite('img/inputs-resized/R_udesa.jpg', R_udesa_resize)
   




## Detección de características

In [24]:
# Inicializar SIFT
sift = cv2.SIFT_create()

def detectar_y_guardar(nombre, img_gray, img_color):
    kp, des = sift.detectAndCompute(img_gray, None)
    print(f"{nombre}: {len(kp)} keypoints, descriptores {des.shape if des is not None else None}")
    # Dibujar keypoints sobre la imagen en color
    img_kp = cv2.drawKeypoints(
        img_color, kp, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
        color=(0, 255, 0)  # verde
    )
    cv2.imwrite(f"img/outputs-4/pre-anms/{nombre}_sift_kp.jpg", img_kp)

    # Serializar keypoints como array numpy
    kparr = np.array([
        (k.pt[0], k.pt[1], k.size, k.angle, k.response, k.octave, k.class_id)
        for k in kp
    ], dtype=np.float32)

    # Guardar kp + descriptores en un archivo comprimido
    np.savez_compressed(f"img/outputs-4/pre-anms/{nombre}_sift_preANMS.npz",
                        keypoints=kparr, descriptors=des)
    return kp, des

# Detectar en todas las imágenes
for nombre, path in [
    ("L_cuadro", "img/inputs/cuadro_0.jpg"),
    ("C_cuadro", "img/inputs/cuadro_1.jpg"),
    ("R_cuadro", "img/inputs/cuadro_2.jpg"),
    ("L_udesa", "img/inputs/udesa_0.jpg"),
    ("C_udesa", "img/inputs/udesa_1.jpg"),
    ("R_udesa", "img/inputs/udesa_2.jpg"),
]:
    img_color = cv2.imread(path, cv2.IMREAD_COLOR)   # para dibujar
    img_gray  = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)  # para SIFT
    detectar_y_guardar(nombre, img_gray, img_color)


L_cuadro: 13634 keypoints, descriptores (13634, 128)
C_cuadro: 15150 keypoints, descriptores (15150, 128)
C_cuadro: 15150 keypoints, descriptores (15150, 128)
R_cuadro: 13945 keypoints, descriptores (13945, 128)
R_cuadro: 13945 keypoints, descriptores (13945, 128)
L_udesa: 22107 keypoints, descriptores (22107, 128)
L_udesa: 22107 keypoints, descriptores (22107, 128)
C_udesa: 14852 keypoints, descriptores (14852, 128)
C_udesa: 14852 keypoints, descriptores (14852, 128)
R_udesa: 5363 keypoints, descriptores (5363, 128)
R_udesa: 5363 keypoints, descriptores (5363, 128)


## Supresión de No Máxima Adaptativa

In [ ]:
import os

PRE_DIR  = "img/outputs-4/pre-anms"
IDX_DIR  = "img/outputs-4/anms-idx"
OUT_DIR  = "img/outputs-4/post-anms"

def anms(keypoints, responses, N):
    K = len(keypoints)
    radii = np.full(K, np.inf)
    for i in range(K):
        for j in range(K):
            if responses[j] > responses[i]:
                dist = np.sqrt((keypoints[j][0]-keypoints[i][0])**2 +
                               (keypoints[j][1]-keypoints[i][1])**2)
                if dist < radii[i]:
                    radii[i] = dist
    indices = np.argsort(-radii)
    return indices[:N]

bases = ["L_cuadro","C_cuadro","R_cuadro",
         "L_udesa","C_udesa","R_udesa"]

N_KEEP = 500  # podés ajustar

for base in bases:
    # Cargar pre-anms
    data = np.load(os.path.join(PRE_DIR, f"{base}_sift_preANMS.npz"))
    kps = data["keypoints"]   # (N,7)
    des = data["descriptors"]

    # Correr ANMS
    coords = kps[:, :2]
    responses = kps[:, 4]
    selected_idx = anms(coords, responses, N_KEEP)

    # Guardar índices
    np.save(os.path.join(IDX_DIR, f"{base}_anms_idx.npy"), selected_idx)

    # Filtrar y guardar post-anms
    kps_f = kps[selected_idx]
    des_f = des[selected_idx]
    np.savez_compressed(os.path.join(OUT_DIR, f"{base}_sift_postANMS.npz"),
                        keypoints=kps_f.astype(np.float32),
                        descriptors=des_f.astype(np.float32))


KeyboardInterrupt: 

## Descripción de caracerísticas

In [ ]:
def rootsift(des):
    des = des.astype(np.float32)
    des /= (des.sum(axis=1, keepdims=True) + 1e-12)
    return np.sqrt(des)

USE_ROOTSIFT = True

for base in bases:
    # Archivos de entrada
    pre_path = os.path.join(PRE_DIR, f"{base}_sift_preANMS.npz")
    idx_path = os.path.join(IDX_DIR, f"{base}_anms_idx.npy")
    out_path = os.path.join(OUT_DIR, f"{base}_sift_postANMS.npz")

    # Cargar datos pre-ANMS
    data = np.load(pre_path)
    kparr_all = data["keypoints"]   # (N,7)
    des_all   = data["descriptors"] # (N,128)

    # Cargar índices seleccionados por ANMS
    idx_keep = np.load(idx_path).astype(np.int32)

    # Filtrar kp y des
    kparr_f = kparr_all[idx_keep]
    des_f   = des_all[idx_keep]

    if USE_ROOTSIFT:
        des_f = rootsift(des_f)

    # Guardar post-ANMS
    np.savez_compressed(out_path,
                        keypoints=kparr_f.astype(np.float32),
                        descriptors=des_f.astype(np.float32))

    

## Asociación de correspondencias (matching)

In [ ]:
# FLANN mejor!
# Flann acelera el matching de desciptores usando estructuras de datos.

# BFMatcher
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)  # or pass empty dictionary

flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des_obj, des_scn, k=2)

# Nos quedamos con matches cuyo segundo mejor match está lejos
matchesMask = [[0, 0] for i in range(len(matches))]
for i, (m, n) in enumerate(matches):
    if m.distance < 0.5 * n.distance:
        matchesMask[i] = [1, 0]

# Dibuja sólo buenos matches
draw_params = dict(matchColor=(0, 255, 0),
                   singlePointColor=(255, 0, 0),
                   matchesMask=matchesMask,
                   flags=cv2.DrawMatchesFlags_DEFAULT)


res_img = cv2.drawMatchesKnn(
    img_obj, kp_obj, img_scn, kp_scn, matches, None, **draw_params
)

plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()

## Eliminación de outliers utilizando RANSAC

## Estimación de la Homografía 

## Juntar y mezclar imágenes